# 实验背景

使用疫情期间的相关[新闻数据](https://covid19.thunlp.org/archives/4/)进行研究

# 实验目标

根据提供的新闻数据，首先对新闻类型进行划分，总共划分为10个类别：体育, 财经, 房产, 家居, 教育, 科技, 时尚, 时政, 游戏, 娱乐
划分新闻后，根据对新闻中对应的评论进行情感分析，分析用户们对新闻的评价总体偏正向或负向。

# 实验流程
下载数据集后解压，将数据集放在当前目录下。

In [ ]:
# !unzip rumor.zip
# !unzip news.zip

Archive:  rumor.zip
   creating: rumor/
  inflating: rumor/fact.json         
   creating: __MACOSX/
   creating: __MACOSX/rumor/
  inflating: __MACOSX/rumor/._fact.json  
  inflating: rumor/.DS_Store         
  inflating: __MACOSX/rumor/._.DS_Store  
   creating: rumor/rumor_forward_comment/
  inflating: rumor/rumor_forward_comment/2020-02-22_K1CaS7gxf7K8j.json  
   creating: __MACOSX/rumor/rumor_forward_comment/
  inflating: __MACOSX/rumor/rumor_forward_comment/._2020-02-22_K1CaS7gxf7K8j.json  
  inflating: rumor/rumor_forward_comment/2020-02-02_K1CaS7Q5i760g.json  
  inflating: __MACOSX/rumor/rumor_forward_comment/._2020-02-02_K1CaS7Q5i760g.json  
  inflating: rumor/rumor_forward_comment/2020-01-22_K1CaS7Qxd76ol.json  
  inflating: __MACOSX/rumor/rumor_forward_comment/._2020-01-22_K1CaS7Qxd76ol.json  
  inflating: rumor/rumor_forward_comment/2020-01-23_K1CaS7Qxg7akd.json  
  inflating: __MACOSX/rumor/rumor_forward_comment/._2020-01-23_K1CaS7Qxg7akd.json  
  inflating: rumor/rumor_fo

选择本次实验使用的数据，位于解压后的news/目录下。

In [ ]:
import os

base_dir = 'news'
comment_dir = os.path.join(base_dir, 'comment')
data_dir = os.path.join(base_dir, 'data')

# List first 5 files in each directory
comment_files = sorted(os.listdir(comment_dir))
data_files = sorted(os.listdir(data_dir))

print(f"Files in {comment_dir} (first 5): {comment_files[:5]}")
print(f"Files in {data_dir} (first 5): {data_files[:5]}")

# Read and display content of the first file from each
def read_file(filepath):
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            return f.read()
    except Exception as e:
        return f"Error reading file: {e}"

if comment_files:
    first_comment_file = os.path.join(comment_dir, comment_files[0])
    print(f"\n--- Content of {first_comment_file} ---")
    print(read_file(first_comment_file)[:500]) # Print first 500 chars

if data_files:
    first_data_file = os.path.join(data_dir, data_files[0])
    print(f"\n--- Content of {first_data_file} ---")
    print(read_file(first_data_file)[:5000]) # Print first 500 chars

Files in news/comment (first 5): ['01-01.txt', '01-02.txt', '01-03.txt', '01-04.txt', '01-05.txt']
Files in news/data (first 5): ['01-01.txt', '01-02.txt', '01-03.txt', '01-04.txt', '01-05.txt']

--- Content of news/comment/01-01.txt ---
[
  {
    "comment": [],
    "time": "01-01 00:00",
    "title": "担忧情绪消散 商品有望迎来2016年来表现最好的一年",
    "url": "https://finance.sina.com.cn/stock/usstock/c/2020-01-01/doc-iihnzahk1229877.shtml"
  },
  {
    "comment": [],
    "time": "01-01 00:01",
    "title": "国泰君安贺青：奋进新时代 共建健康繁荣的资本市场",
    "url": "https://finance.sina.com.cn/stock/roll/2020-01-01/doc-iihnzahk1237906.shtml"
  },
  {
    "comment": [],
    "time": "01-01 00:02",
    "title": "新京报社论：穿越大江大河",
    "url": "https://news.sina.com.cn/c/202

--- Content of news/data/01-01.txt ---
[
  {
    "meta": {
      "content": "大宗商品将创下自2016年以来的最佳年度表现，原油到铜的年度收益将有所增长。 彭博商品指数创下自2018年11月以来的最高水平，原因是贸易紧张局势缓解，风险偏好情绪席卷市场，以及美元走软。该指数在2019年已经上涨11%。 大宗商品正受益于年底的上涨，因为至少在目前，2020年的前景似乎比今年大部分时间的情况更有希望。 华侨银行经济学家Howie Lee表示，到2

## 数据加载与预处理

将数据集内容分别从`data`和`comments`目录加载，并合并到一个DataFrame中。


In [29]:
import os
import json
import pandas as pd

# Define directories (re-using variables from context if available, but defining for clarity)
data_dir = 'news/data'
comment_dir = 'news/comment'

# Get sorted list of filenames from data_dir
data_files = sorted(os.listdir(data_dir))

records = []

for filename in data_files:
    # Construct full paths
    data_filepath = os.path.join(data_dir, filename)
    comment_filepath = os.path.join(comment_dir, filename)

    # Check if corresponding comment file exists
    if os.path.exists(comment_filepath):
        try:
            with open(data_filepath, 'r', encoding='utf-8') as f_data, \
                 open(comment_filepath, 'r', encoding='utf-8') as f_comment:

                data_items = json.load(f_data)
                comment_items = json.load(f_comment)

                # Iterate through items assuming lists are parallel and sorted by index implicitly
                # Using zip to iterate through both lists simultaneously
                for d_item, c_item in zip(data_items, comment_items):
                    content = d_item.get('meta', {}).get('content', '')
                    comments = c_item.get('comment', [])

                    records.append({
                        'content': content,
                        'comments': comments
                    })
        except Exception as e:
            print(f"Error processing {filename}: {e}")

# Create DataFrame
df = pd.DataFrame(records)

# Display DataFrame info
print("DataFrame Head:")
display(df.head())
print("\nDataFrame Info:")
print(df.info())

DataFrame Head:


,content,comments
0,大宗商品将创下自2016年以来的最佳年度表现，原油到铜的年度收益将有所增长。 彭博商品指数创...,[]
1,国泰君安党委书记贺青：奋进新时代 共建持续健康繁荣的资本市场 贺青 国泰君安证券 “党的十九...,[]
2,原标题：穿越大江大河 时光不恋过往，大江永远奔腾。你好，2020年！你好，下一个十年！ 当2...,[]
3,新浪美股1月1日讯 美国总统特朗普曾经强调，被新减税政策选中的低收入地区房价“飙升”，证明他...,"[{'area': '福建泉州', 'content': '看看消费水平就知道了，啥也别说了..."
4,原标题：新年前一天，林郑在警总告诉警队：相信正义站在我们一边！ 林郑月娥说，香港这支卓越的警...,"[{'area': '湖北黄石', 'content': '香港的明天会更加美好！', 'n..."



DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 129969 entries, 0 to 129968
Data columns (total 2 columns):
 #   Column    Non-Null Count   Dtype 
---  ------    --------------   ----- 
 0   content   129969 non-null  object
 1   comments  129969 non-null  object
dtypes: object(2)
memory usage: 2.0+ MB
None


读取新闻总数129969条，并且与评论(若存在)对应。

## 任务一: 新闻类别划分
将所有收集到的新闻按照10类进行划分，以便能根据类别筛选出新闻，具体划分类别如下:
1. 财经
2. 科技
3. 娱乐
4. 体育
5. 时尚
6. 教育
7. 时政
8. 家居
9. 房产
10. 游戏

## 新闻类别划分模型准备
新闻类别需要根据新闻主体内的文字解析，并对新闻的类别进行判断，参考[中文文本分类实现](https://github.com/gaussic/text-classification-cnn-rnn/tree/master),基于[THUCTC: 一个高效的中文文本分类工具包](http://thuctc.thunlp.org/)的数据集的子集构建，训练与测试后得到的模型进行分类任务。



将数据集与主要相关程序下载到`text_classification` 文件夹

In [ ]:
import os

# Define the path to the text_classification folder
text_classification_path = 'text_classification'

# List contents of the folder
if os.path.exists(text_classification_path):
    contents = os.listdir(text_classification_path)
    print(f"Contents of '{text_classification_path}':")
    for item in contents:
        print(item)
else:
    print(f"The folder '{text_classification_path}' does not exist.")

Contents of 'text_classification':
cnews_loader.py
data
run_cnn.py
predict.py
__pycache__
cnn_model.py


数据集位置定位

In [ ]:
import os

data_folder_path = os.path.join(text_classification_path, 'data')

if os.path.exists(data_folder_path):
    data_contents = os.listdir(data_folder_path)
    print(f"Contents of '{data_folder_path}':")
    for item in data_contents:
        print(item)
else:
    print(f"The folder '{data_folder_path}' does not exist.")

Contents of 'text_classification/data':
cnews.vocab.txt
cnews.val.txt
cnews.test.txt
cnews.train.txt


### 数据集加载

In [ ]:
import sys
import os

# Add the text_classification directory to sys.path to import cnews_loader
text_classification_path = 'text_classification'
if text_classification_path not in sys.path:
    sys.path.insert(0, text_classification_path)

import cnews_loader as text_loader

# Define data file paths
base_data_path = os.path.join(text_classification_path, 'data')
vocab_path = os.path.join(base_data_path, 'cnews.vocab.txt')
train_path = os.path.join(base_data_path, 'cnews.train.txt')
val_path = os.path.join(base_data_path, 'cnews.val.txt')
test_path = os.path.join(base_data_path, 'cnews.test.txt')

# Define parameters
max_length = 600 # This is a default value in process_file, explicitly setting for clarity

# Load vocabulary and categories
words, word_to_id = text_loader.read_vocab(vocab_path)
categories, cat_to_id = text_loader.read_category()

# Load datasets
x_train, y_train = text_loader.process_file(train_path, word_to_id, cat_to_id, max_length)
x_val, y_val = text_loader.process_file(val_path, word_to_id, cat_to_id, max_length)
x_test, y_test = text_loader.process_file(test_path, word_to_id, cat_to_id, max_length)

print(f"Vocabulary size: {len(words)}")
print(f"Number of categories: {len(categories)}")
print(f"Shape of x_train: {x_train.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of x_val: {x_val.shape}")
print(f"Shape of y_val: {y_val.shape}")
print(f"Shape of x_test: {x_test.shape}")
print(f"Shape of y_test: {y_test.shape}")


Vocabulary size: 5000
Number of categories: 10
Shape of x_train: (50000, 600)
Shape of y_train: (50000, 10)
Shape of x_val: (5000, 600)
Shape of y_val: (5000, 10)
Shape of x_test: (10000, 600)
Shape of y_test: (10000, 10)


### 模型训练



#### 修改 `cnn_model.py` 内容

由于该项目最后更新时间距今已经7年，其中所使用的某些程序由于资源包更新已经被停用，于是下面基于类似的模型构建逻辑以现版本支持的方式重新搭建模型。


In [ ]:
# import os

# keras_model_content = '''# coding: utf-8

# import tensorflow as tf
# from tensorflow.keras import layers, Model

# class TCNNConfig(object):
#     """CNN配置参数"""

#     embedding_dim = 64  # 词向量维度
#     seq_length = 600  # 序列长度
#     num_classes = 10  # 类别数
#     num_filters = 256  # 卷积核数目
#     kernel_size = 5  # 卷积核尺寸
#     vocab_size = 5000  # 词汇表达小

#     hidden_dim = 128  # 全连接层神经元

#     dropout_keep_prob = 0.5  # dropout保留比例
#     learning_rate = 1e-3  # 学习率

#     batch_size = 64  # 每批训练大小
#     num_epochs = 10  # 总迭代轮次

#     print_per_batch = 100  # 每多少轮输出一次结果
#     save_per_batch = 10  # 每多少轮存入tensorboard


# class TextCNN(Model):
#     """文本分类，CNN模型"""

#     def __init__(self, config):
#         super(TextCNN, self).__init__()
#         self.config = config

#         self.embedding = layers.Embedding(config.vocab_size, config.embedding_dim)
#         self.conv = layers.Conv1D(config.num_filters, config.kernel_size, activation='relu')
#         self.gmp = layers.GlobalMaxPooling1D()
#         self.fc1 = layers.Dense(config.hidden_dim, activation='relu')
#         self.dropout = layers.Dropout(1 - config.dropout_keep_prob) # Keras dropout uses drop rate
#         self.fc2 = layers.Dense(config.num_classes, activation='softmax')

#     def call(self, inputs):
#         x = self.embedding(inputs)
#         x = self.conv(x)
#         x = self.gmp(x)
#         x = self.fc1(x)
#         x = self.dropout(x)
#         return self.fc2(x)


# '''

# cnn_model_path = os.path.join('text_classification', 'cnn_model.py')

# # Write the modified content back
# with open(cnn_model_path, 'w', encoding='utf-8') as f:
#     f.write(keras_model_content)

# print(f"Modified '{cnn_model_path}' to use Keras API for TextCNN class.")

Modified 'text_classification/cnn_model.py' to use Keras API for TextCNN class.


#### 进行训练
使用现版本tensorflow支持的程序进行训练，并保存模型参数。

In [ ]:
import tensorflow as tf
import time
from datetime import timedelta
import os
import sys

# Ensure text_classification is in sys.path for imports
text_classification_path = 'text_classification'
if text_classification_path not in sys.path:
    sys.path.insert(0, text_classification_path)

In [ ]:


# Re-import TextCNN from the modified cnn_model.py
from cnn_model import TCNNConfig, TextCNN

# Define the config for the model, ensuring parameters match previously loaded data
config = TCNNConfig()
config.vocab_size = len(words) # Use the actual vocabulary size
config.seq_length = max_length # Use the actual max_length
config.num_classes = len(categories)

# Define model saving paths
save_dir = os.path.join('text_classification', 'checkpoints/textcnn')
if not os.path.exists(save_dir):
    os.makedirs(save_dir)
save_path = os.path.join(save_dir, 'best_validation.weights.h5') # Save in Keras .h5 format

print("Configuring and training Keras model...")

# Instantiate the Keras model
keras_model = TextCNN(config)

# Compile the model
keras_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=config.learning_rate),
    loss=tf.keras.losses.CategoricalCrossentropy(),
    metrics=['accuracy']
)

# Setup ModelCheckpoint callback
checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=save_path,
    monitor='val_accuracy',
    save_best_only=True,
    save_weights_only=True, # Save the whole model
    verbose=1
)

start_time = time.time()

# Train the model
history = keras_model.fit(
    x_train, y_train,
    batch_size=config.batch_size,
    epochs=config.num_epochs,
    validation_data=(x_val, y_val),
    callbacks=[checkpoint_callback]
)

end_time = time.time()
time_dif = timedelta(seconds=int(round(end_time - start_time)))
print("\nTraining complete. Time usage:", time_dif)

# Evaluate the model on test data (optional, but good practice)
print("\nEvaluating on test data...")
loss, accuracy = keras_model.evaluate(x_test, y_test, batch_size=config.batch_size)
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.2%}")

Configuring and training Keras model...
Epoch 1/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7442 - loss: 0.8266
Epoch 1: val_accuracy improved from -inf to 0.90980, saving model to text_classification/checkpoints/textcnn/best_validation.weights.h5
782/782 ━━━━━━━━━━━━━━━━━━━━ 9s 8ms/step - accuracy: 0.7444 - loss: 0.8260 - val_accuracy: 0.9098 - val_loss: 0.3194
Epoch 2/10
779/782 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9667 - loss: 0.1104
Epoch 2: val_accuracy improved from 0.90980 to 0.93720, saving model to text_classification/checkpoints/textcnn/best_validation.weights.h5
782/782 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9667 - loss: 0.1104 - val_accuracy: 0.9372 - val_loss: 0.2145
Epoch 3/10
776/782 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9861 - loss: 0.0506
Epoch 3: val_accuracy did not improve from 0.93720
782/782 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9861 - loss: 0.0506 - val_accuracy: 0.9336 - val_loss: 0.2166
Epoch 4/10
780/782 ━━━━━━

模型在经过10个epochs之后在训练用的数据集上达到了0.9991的准确率，同时在测试集上也达到了0.9621的准确率，效果还是相当理想的。并且将训练过程中在验证集上表现最好的模型参数保存至`checkpoints/textcnn`中。

下面是模型表现效果进一步可视化的展示:

In [ ]:
def get_time_dif(start_time):
    """获取已使用时间"""
    end_time = time.time()
    time_dif = end_time - start_time
    return timedelta(seconds=int(round(time_dif)))

In [ ]:
from sklearn import metrics
import numpy as np
import time
from datetime import timedelta

def test_keras_model(model, x_test, y_test, categories, config):
    print("Loading test data...")
    start_time = time.time()

    # Evaluate the Keras model
    print('Testing Keras model...')
    loss_test, acc_test = model.evaluate(x_test, y_test, batch_size=config.batch_size, verbose=0)
    msg = 'Test Loss: {0:>6.2}, Test Acc: {1:>7.2%}'
    print(msg.format(loss_test, acc_test))

    # Make predictions
    y_pred_probabilities = model.predict(x_test, batch_size=config.batch_size, verbose=0)
    y_pred_cls = np.argmax(y_pred_probabilities, axis=1)

    # Get true class labels
    y_test_cls = np.argmax(y_test, axis=1)

    # Evaluate using sklearn metrics
    print("\nPrecision, Recall and F1-Score...")
    print(metrics.classification_report(y_test_cls, y_pred_cls, target_names=categories))

    print("Confusion Matrix...")
    cm = metrics.confusion_matrix(y_test_cls, y_pred_cls)
    print(cm)

    time_dif = timedelta(seconds=int(round(time.time() - start_time)))
    print("\nTime usage:", time_dif)

# Assuming loaded_model, x_test, y_test, categories, and config are available in the global scope
# Call the new test function
test_keras_model(loaded_model, x_test, y_test, categories, config)

Loading test data...
Testing Keras model...
Test Loss:   0.16, Test Acc:  96.16%

Precision, Recall and F1-Score...
              precision    recall  f1-score   support

          体育       0.99      1.00      1.00      1000
          财经       0.92      0.99      0.96      1000
          房产       1.00      1.00      1.00      1000
          家居       0.97      0.89      0.93      1000
          教育       0.94      0.92      0.93      1000
          科技       0.93      0.98      0.96      1000
          时尚       0.96      0.97      0.96      1000
          时政       0.96      0.94      0.95      1000
          游戏       1.00      0.94      0.97      1000
          娱乐       0.95      0.98      0.96      1000

    accuracy                           0.96     10000
   macro avg       0.96      0.96      0.96     10000
weighted avg       0.96      0.96      0.96     10000

Confusion Matrix...
[[996   1   0   1   1   0   0   0   0   1]
 [  0 993   0   0   1   2   0   4   0   0]
 [  0   1 997   1  

结果可见模型的表现无论是精准率还是召回率都相当好，召回率最差的家居为0.89。

#### 模型加载与预测
将模型参数从checkpoint/textcnn中加载，并再次在测试集上预测。下面分别测试了加载参数前后的表现，通过模型表现确认模型参数加载成功。

In [ ]:
import tensorflow as tf
import os
from cnn_model import TextCNN, TCNNConfig # Ensure your custom model class is imported

# Define the path to your saved model
save_dir = os.path.join('text_classification', 'checkpoints/textcnn')
checkpoint_path = os.path.join(save_dir, 'best_validation.weights.h5')


# Define the config for the model, ensuring parameters match previously loaded data
config = TCNNConfig()
config.vocab_size = len(words) # Use the actual vocabulary size
config.seq_length = max_length # Use the actual max_length
config.num_classes = len(categories)

# # Define model saving paths
# save_dir = os.path.join('text_classification', 'checkpoints/textcnn')
# if not os.path.exists(save_dir):
#     os.makedirs(save_dir)
# save_path = os.path.join(save_dir, 'best_validation.weights.h5') # Save in Keras .h5 format

# print("Configuring and training Keras model...")

# Instantiate the Keras model
loaded_model = TextCNN(config)
loaded_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=config.learning_rate),
    loss=tf.keras.losses.CategoricalCrossentropy(),
    metrics=['accuracy']
)
loss, accuracy = loaded_model.evaluate(x_test, y_test, batch_size=config.batch_size)
print("Untrained model, accuracy: {:5.2f}%".format(100 * accuracy))

loaded_model.load_weights(checkpoint_path)
loss, accuracy = loaded_model.evaluate(x_test, y_test, batch_size=config.batch_size)
print("Restored model, accuracy: {:5.2f}%".format(100 * accuracy))



157/157 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - accuracy: 0.1157 - loss: 2.3014
Untrained model, accuracy: 10.43%
 60/157 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9798 - loss: 0.0861

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 16 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9669 - loss: 0.1342
Restored model, accuracy: 96.16%


#### 对新闻数据分类

准备好并顺利加载模型后，下面开始使用加载的模型在新闻数据集上对新闻进行分类。

In [ ]:
import numpy as np
import sys
import os

# Add the text_classification directory to sys.path to import cnews_loader
text_classification_path = 'text_classification'
if text_classification_path not in sys.path:
    sys.path.insert(0, text_classification_path)

import cnews_loader as text_loader

def preprocess_texts_for_prediction(texts, word_to_id, max_length):
    """Preprocesses a list of text strings for model prediction."""
    data_id = []
    for content in texts:
        # Convert content string to list of characters
        char_list = list(text_loader.native_content(content))
        # Map characters to word IDs, filtering out unknown words
        data_id.append([word_to_id[x] for x in char_list if x in word_to_id])

    # Pad sequences to a fixed length
    x_pad = tf.keras.preprocessing.sequence.pad_sequences(data_id, max_length)
    return x_pad

print("Preprocessing news content for classification...")
# Preprocess the 'content' column of df
x_news = preprocess_texts_for_prediction(df['content'].tolist(), word_to_id, max_length)

print("Classifying news articles using the loaded model...")
# Make predictions with the loaded model
y_pred_probabilities = loaded_model.predict(x_news)

# Convert probabilities to class indices
y_pred_indices = np.argmax(y_pred_probabilities, axis=1)

# Map class indices to category names
df['news_category'] = [categories[idx] for idx in y_pred_indices]

print("News classification complete. Displaying first 10 rows with categories:")
display(df.head(10))

Preprocessing news content for classification...
Classifying news articles using the loaded model...
4062/4062 ━━━━━━━━━━━━━━━━━━━━ 7s 1ms/step
News classification complete. Displaying first 5 rows with categories:


,content,comments,news_category
0,大宗商品将创下自2016年以来的最佳年度表现，原油到铜的年度收益将有所增长。 彭博商品指数创...,[],财经
1,国泰君安党委书记贺青：奋进新时代 共建持续健康繁荣的资本市场 贺青 国泰君安证券 “党的十九...,[],财经
2,原标题：穿越大江大河 时光不恋过往，大江永远奔腾。你好，2020年！你好，下一个十年！ 当2...,[],教育
3,新浪美股1月1日讯 美国总统特朗普曾经强调，被新减税政策选中的低收入地区房价“飙升”，证明他...,"[{'area': '福建泉州', 'content': '看看消费水平就知道了，啥也别说了...",科技
4,原标题：新年前一天，林郑在警总告诉警队：相信正义站在我们一边！ 林郑月娥说，香港这支卓越的警...,"[{'area': '湖北黄石', 'content': '香港的明天会更加美好！', 'n...",时政
5,原标题：驻伊拉克使馆被“攻破”，特朗普大骂伊朗！ 特朗普：伊朗正在策划对美国驻伊拉克大使馆的...,[],教育
6,原标题：创新高！2019年中国内地票房642.66亿元 17.27亿人次观影 中新经纬客户端...,"[{'area': '广东广州', 'content': '我和我的祖国，除了包场和刷票，估...",娱乐
7,原标题：创新高！2019年中国内地票房642.66亿元 17.27亿人次观影 中新经纬客户端...,[],娱乐
8,原标题：2020年起，这些新规正式施行，“医”食住行都涵盖 中新经纬客户端1月1日电 （熊思...,"[{'area': '湖南', 'content': '2020演唱会🙏', 'nickna...",时政
9,原标题：2019年人民币汇率中间价调贬1130基点 今年或小幅升值 中新经纬客户端1月1日电...,"[{'area': '香港', 'content': '赶紧破8，不然就来不及了。', 'n...",财经


## 任务2:评论情感分析
通过分析新闻的评论，用以初步判断大众对新闻的观感。

## 评论展示

随机挑选一则新闻，并逐行展示其所有的评论。


In [ ]:
print("Comments for the fourth row (index 3):")
for i in range(len(df.loc[4, 'comments'])):
  print(df.loc[4, 'comments'][i]['content'])

Comments for the fourth row (index 3):
香港的明天会更加美好！
港警你们辛苦了，祝我们伟大的港警节日快乐乐！
再赞
👍
加油啊，香港警察！
卓越的警方
全球最辛苦的工作就是警察
警队很棒！教育司法界应严格整束！
香港
哈哈
港警哥辛苦了，完成任务后来青岛喝啤酒。
支持！
厉害
像中国一样平安
支持警察
八十年代的剧集里的台词：我们HongKong policman 二十四小时on duty！
嗯嗯
正义永存！
加油！！！！
遏制暴力维护法纪
支持
好
👍🏻
加油
负重前行
加油
要有一支能打胜仗的队伍，
真的不错
大力支持
。。
负重前行！
香港稳定的重要力量。
哦？
👍
很酷
稳定的支柱
真棒
支持
好
开动脑筋动些真格
赞一个
说得好
哈哈哈用人朝前啊
支持
少跟那些受国外势力支助，拿到国外钱的恐怖分子废话，全部清理出来拉去枪🔫毙了[抓狂]
1
香港警队纪律严明
支持港警
啊
2020一切都好
正义必将战胜邪恶！
希望早点结束风波
致敬！
👍
👍
兵哥哥
支持支持[赞]
好
👍
相信正义！


### 模型准备

本次情感分析模型使用来自带情感标注的[微博数据集](https://github.com/SophonPlus/ChineseNlpCorpus/blob/master/datasets/weibo_senti_100k/intro.ipynb?short_path=9e00bbc)进行训练，该数据集有着10万条明确地以1和0表示正向与负向的评论，将数据集下载后并放置在sentimental_analysis_v1文件夹中。

#### 数据集加载

In [ ]:
import pandas as pd
import os

sentimental_analysis_path = 'sentimental_analysis_v1'
csv_filepath = os.path.join(sentimental_analysis_path, 'weibo_senti_100k.csv')

# Check if the file exists and load it into a DataFrame
if os.path.exists(csv_filepath):
    df_sentiment = pd.read_csv(csv_filepath)
    print(f"Successfully loaded '{csv_filepath}'.")
    print("\nDataFrame Head:")
    print(df_sentiment.head())
    print("\nDataFrame Info:")
    df_sentiment.info()
    print("\nValue counts for 'label' column:")
    print(df_sentiment['label'].value_counts())
else:
    print(f"The file '{csv_filepath}' does not exist.")

Successfully loaded 'sentimental_analysis_v1/weibo_senti_100k.csv'.

DataFrame Head:
   label                                             review
0      1              ﻿更博了，爆照了，帅的呀，就是越来越爱你！生快傻缺[爱你][爱你][爱你]
1      1  @张晓鹏jonathan 土耳其的事要认真对待[哈哈]，否则直接开除。@丁丁看世界 很是细心...
2      1  姑娘都羡慕你呢…还有招财猫高兴……//@爱在蔓延-JC:[哈哈]小学徒一枚，等着明天见您呢/...
3      1                                         美~~~~~[爱你]
4      1                                  梦想有多大，舞台就有多大![鼓掌]

DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119988 entries, 0 to 119987
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   label   119988 non-null  int64 
 1   review  119988 non-null  object
dtypes: int64(1), object(1)
memory usage: 1.8+ MB

Value counts for 'label' column:
label
0    59995
1    59993
Name: count, dtype: int64


#### 模型选用
本次情感分析任务选用适用于中文的预训练transformer模型`bert-base-chinese`并基于该模型做迁移训练，下面先从huggingface下载transformers库，以及其他必要的库。

In [ ]:
# !pip install transformers datasets evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.9 MB/s eta 0:00:00


## 预训练模型 Tokenizer and Model加载



In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

# 2. Define the name of a suitable pre-trained Chinese Transformer model
model_name = "bert-base-chinese"

# 3. Load the tokenizer for the chosen model
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 4. Load the pre-trained model for sequence classification, specifying num_labels for binary classification
# Assuming 2 labels (positive/negative) for sentiment analysis
num_labels_sentiment = 2
model_sentiment = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels_sentiment)

# 5. Print a confirmation message
print(f"Successfully loaded tokenizer and model '{model_name}' for sequence classification with {num_labels_sentiment} labels.")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-chinese and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Successfully loaded tokenizer and model 'bert-base-chinese' for sequence classification with 2 labels.


In [ ]:
import numpy as np

# Tokenize the 'review' column
# This will return a dictionary with 'input_ids', 'token_type_ids', and 'attention_mask'
# We will use padding and truncation to handle variable sequence lengths.
# max_length is chosen to be representative of typical review lengths, or can be determined dynamically.
tokenized_sentiment_data = tokenizer(df_sentiment['review'].tolist(),
                                     padding=True,
                                     truncation=True,
                                     max_length=128, # A common max length for BERT-like models for short texts
                                     return_tensors='pt') # Return PyTorch tensors

# Convert labels to PyTorch tensors
labels_sentiment = torch.tensor(df_sentiment['label'].tolist())

print("Tokenization complete for sentiment analysis data.")
print(f"Shape of input_ids: {tokenized_sentiment_data['input_ids'].shape}")
print(f"Shape of attention_mask: {tokenized_sentiment_data['attention_mask'].shape}")
print(f"Shape of labels: {labels_sentiment.shape}")

Tokenization complete for sentiment analysis data.
Shape of input_ids: torch.Size([119988, 128])
Shape of attention_mask: torch.Size([119988, 128])
Shape of labels: torch.Size([119988])


#### 开始迁移训练

下面开始基于微博评论数据集对模型进行训练，模型参数保存`fine_tuned_bert_sentiment`中。

In [ ]:
from transformers import TrainingArguments, Trainer
import torch
from sklearn.model_selection import train_test_split

In [ ]:


# 2. Create a custom PyTorch Dataset class
class SentimentDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

# Split the tokenized data and labels into training and validation sets
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df_sentiment['review'].tolist(),
    df_sentiment['label'].tolist(),
    test_size=0.1,  # Use 10% of data for validation
    random_state=42,
    stratify=df_sentiment['label'].tolist()
)

# Tokenize the split data
tokenized_train_sentiment_data = tokenizer(train_texts,
                                     padding=True,
                                     truncation=True,
                                     max_length=128,
                                     return_tensors='pt')

tokenized_val_sentiment_data = tokenizer(val_texts,
                                     padding=True,
                                     truncation=True,
                                     max_length=128,
                                     return_tensors='pt')

# Convert labels to PyTorch tensors
labels_train_sentiment = torch.tensor(train_labels)
labels_val_sentiment = torch.tensor(val_labels)

# 3. Instantiate the SentimentDataset for the training and validation data
train_dataset_sentiment = SentimentDataset(tokenized_train_sentiment_data, labels_train_sentiment)
eval_dataset_sentiment = SentimentDataset(tokenized_val_sentiment_data, labels_val_sentiment)

In [ ]:


# Define output directory for the fine-tuned model and tokenizer
sentiment_model_output_dir = os.path.join(sentimental_analysis_path, 'fine_tuned_bert_sentiment')
os.makedirs(sentiment_model_output_dir, exist_ok=True)

# 4. Define TrainingArguments for the fine-tuning process
training_args = TrainingArguments(
    output_dir=os.path.join(sentimental_analysis_path, 'results'),  # output directory
    num_train_epochs=3,                  # total number of training epochs
    per_device_train_batch_size=16,      # batch size per device during training
    per_device_eval_batch_size=64,       # batch size for evaluation
    warmup_steps=500,                    # number of warmup steps for learning rate scheduler
    weight_decay=0.01,                   # strength of weight decay
    logging_dir=os.path.join(sentimental_analysis_path, 'logs'), # directory for storing logs
    logging_steps=500,                   # log metrics every N steps
    eval_strategy="epoch",         # Evaluate at the end of each epoch
    save_strategy="epoch",               # Save checkpoint at the end of each epoch
    load_best_model_at_end=True,         # Load the best model at the end of training
    metric_for_best_model="eval_loss",   # Metric to monitor for best model
    report_to="none"                     # Disable Weights & Biases logging
)

# 5. Initialize the Trainer
trainer = Trainer(
    model=model_sentiment,                         # the instantiated f917 Transformers model to be trained
    args=training_args,                  # training arguments, defined above
    train_dataset=train_dataset_sentiment, # training dataset
    eval_dataset=eval_dataset_sentiment,   # validation dataset
    tokenizer=tokenizer                  # tokenizer to be used
)

# 6. Start the training process
print("Starting sentiment model fine-tuning...")
trainer.train()
print("Sentiment model fine-tuning complete.")

# 7. After training, save the fine-tuned model and tokenizer
trainer.save_model(sentiment_model_output_dir)
tokenizer.save_pretrained(sentiment_model_output_dir)

print(f"Fine-tuned model and tokenizer saved to {sentiment_model_output_dir}")

/tmp/ipython-input-170496554.py:71: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Starting sentiment model fine-tuning...


/tmp/ipython-input-170496554.py:12: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
/tmp/ipython-input-170496554.py:13: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item['labels'] = torch.tensor(self.labels[idx])


Epoch,Training Loss,Validation Loss
1,0.166400,0.132546
2,0.157800,0.142610
3,0.138500,0.134306


/tmp/ipython-input-170496554.py:12: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
/tmp/ipython-input-170496554.py:13: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item['labels'] = torch.tensor(self.labels[idx])
/tmp/ipython-input-170496554.py:12: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
/tmp/ipython-input-170496554.py:13: UserWarning: To copy construct from a tensor, it is recommende

Sentiment model fine-tuning complete.
Fine-tuned model and tokenizer saved to sentimental_analysis_v1/fine_tuned_bert_sentiment


将存储的情感分析模型读取，在划分出的验证集上测试，并输出表现数据。

In [ ]:
import os
import sys
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# Ensure text_classification is in sys.path for imports
text_classification_path = 'text_classification'
if text_classification_path not in sys.path:
    sys.path.insert(0, text_classification_path)

# 2. Create a custom PyTorch Dataset class
class SentimentDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

# Define paths (re-using variables from previous steps)
sentimental_analysis_path = 'sentimental_analysis_v1'
sentiment_model_output_dir = os.path.join(sentimental_analysis_path, 'fine_tuned_bert_sentiment')

print(f"Loading fine-tuned model from {sentiment_model_output_dir}...")

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(sentiment_model_output_dir)

# Load the fine-tuned model
loaded_sentiment_model = AutoModelForSequenceClassification.from_pretrained(sentiment_model_output_dir)

print("Fine-tuned model loaded successfully.")

# Re-split the data and create eval_dataset_sentiment
# Assuming df_sentiment is available from previous execution
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df_sentiment['review'].tolist(),
    df_sentiment['label'].tolist(),
    test_size=0.1,
    random_state=42,
    stratify=df_sentiment['label'].tolist()
)

tokenized_val_sentiment_data = tokenizer(val_texts,
                                     padding=True,
                                     truncation=True,
                                     max_length=128,
                                     return_tensors='pt')
labels_val_sentiment = torch.tensor(val_labels)

eval_dataset_sentiment = SentimentDataset(tokenized_val_sentiment_data, labels_val_sentiment)

# Define TrainingArguments for evaluation
training_args = TrainingArguments(
    output_dir=os.path.join(sentimental_analysis_path, 'results'), # Output directory (can be arbitrary for evaluation only)
    per_device_eval_batch_size=64,
    report_to="none"
)

print("Evaluating loaded model on the validation set...")
# Create a new Trainer instance for evaluation with the loaded model
evaluator = Trainer(
    model=loaded_sentiment_model,
    args=training_args,
    eval_dataset=eval_dataset_sentiment,
    tokenizer=tokenizer # Pass the tokenizer to the trainer
)

# Perform evaluation
metrics = evaluator.evaluate()

print("Evaluation Results:")
for key, value in metrics.items():
    print(f"  {key}: {value:.4f}")

# --- Additional metrics: Accuracy, Precision, Recall, F1-score ---
print("\nGenerating predictions for detailed classification report...")
predictions = evaluator.predict(eval_dataset_sentiment)

# Get predicted labels (argmax of logits)
predicted_labels = predictions.predictions.argmax(axis=1)

# Get true labels
true_labels = predictions.label_ids

# Generate classification report
report = classification_report(true_labels, predicted_labels, target_names=['Negative (0)', 'Positive (1)'])
print("\nClassification Report:")
print(report)


Loading fine-tuned model from sentimental_analysis_v1/fine_tuned_bert_sentiment...
Fine-tuned model loaded successfully.
Evaluating loaded model on the validation set...


/tmp/ipython-input-883958432.py:72: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  evaluator = Trainer(
/tmp/ipython-input-883958432.py:23: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
/tmp/ipython-input-883958432.py:24: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item['labels'] = torch.tensor(self.labels[idx])


Evaluation Results:
  eval_loss: 0.1325
  eval_model_preparation_time: 0.0026
  eval_runtime: 40.0873
  eval_samples_per_second: 299.3220
  eval_steps_per_second: 4.6900

Generating predictions for detailed classification report...

Classification Report:
              precision    recall  f1-score   support

Negative (0)       0.97      0.97      0.97      6000
Positive (1)       0.97      0.97      0.97      5999

    accuracy                           0.97     11999
   macro avg       0.97      0.97      0.97     11999
weighted avg       0.97      0.97      0.97     11999



由结果来看，精准率与召回率都达到了0.97，这个效果相当理想，下面随机罗列出预测正确与预测错误的几个样例。

In [ ]:
# Combine validation texts, true labels, and predicted labels into a DataFrame for easier analysis
results_df = pd.DataFrame({
    'review': val_texts,
    'true_label': true_labels,
    'predicted_label': predicted_labels
})

# Map numerical labels back to meaningful names if desired (0: Negative, 1: Positive)
label_map = {0: 'Negative', 1: 'Positive'}
results_df['true_label_name'] = results_df['true_label'].map(label_map)
results_df['predicted_label_name'] = results_df['predicted_label'].map(label_map)

# Filter for correct predictions
correct_predictions = results_df[results_df['true_label'] == results_df['predicted_label']]

# Filter for incorrect predictions
incorrect_predictions = results_df[results_df['true_label'] != results_df['predicted_label']]

print("\n--- Random Examples of CORRECT Predictions ---")
if not correct_predictions.empty:
    display(correct_predictions.sample(n=min(5, len(correct_predictions)), random_state=42))
else:
    print("No correct predictions found in the sample.")

print("\n--- Random Examples of INCORRECT Predictions ---")
if not incorrect_predictions.empty:
    display(incorrect_predictions.sample(n=min(5, len(incorrect_predictions)), random_state=42))
else:
    print("No incorrect predictions found in the sample.")


--- Random Examples of CORRECT Predictions ---


,review,true_label,predicted_label,true_label_name,predicted_label_name
8743,回复@迷你_黑洞: 我想起「?吹草低?牛羊」[哈哈][哈哈][哈哈] //@迷你_黑洞:回复...,1,1,Positive,Positive
10469,回复@JoliePan:哈哈哈 //@JoliePan:需要个秘书专门可以帮忙做PPT的[抓...,0,0,Negative,Negative
4892,刚想说，有点象american idol的手法，但是真没新意，继续看美剧吧[嘻嘻],1,1,Positive,Positive
11250,落跑新郎是吗[晕]//@全图解:[哈哈],0,0,Negative,Negative
6417,[哈哈][哈哈][哈哈][哈哈][哈哈][哈哈][哈哈]//@段子之父:→_→[哈哈]//@...,1,1,Positive,Positive



--- Random Examples of INCORRECT Predictions ---


,review,true_label,predicted_label,true_label_name,predicted_label_name
7441,哇[泪]谢谢性感的泰国美妞给我的祝福[爱你][爱你][爱你][爱你]珍藏??????,1,0,Positive,Negative
1501,//@兔兔陈小姐：果真是重口味啊~~~//@曹操的爷爷: 哦！！粘土有张力！！//@破碎女王...,0,1,Negative,Positive
9808,生鲜电商，又来一只大象！//@杜非: 如果是POP，还不算有冲击力；如果是自营，那就厉害了―...,0,1,Negative,Positive
6387,大半夜才看到~~泪撒……[泪][泪]……天天家族和天天兄弟也会一直一起的~~[心][爱你]~...,1,0,Positive,Negative
2002,我这哪里是搅和，我对特供机的品质没有任何怀疑，华为的质量是绝对过关的，华勤的ODM也是非常出...,0,1,Negative,Positive


从错误的例子来看还是挺有意思的，6387列的样例是一种以"泪"等含有哭泣因素的形式来表示开心，而2002列的样例咋看下是以看似正向的形式表示负向的情绪，这些较为复杂的情绪表达形式确实也比较难以通过模型精准地判断倾向性。

#### 对新闻评论进行判断

下面将创建一个函数，该函数使用预加载的 `tokenizer` 和 `loaded_sentiment_model` 来预测评论的情感，然后将此函数应用到 `df` DataFrame 中的每个新闻文章，并添加这些评论的数目和平均情感作为新的列 'num_comments' 和 'avg_comment_sentiment'，最后显示更新后的 `df`。

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np

def predict_sentiment_batch(texts, tokenizer, model, max_length=128):
    """Predicts the sentiment (positive probability) for a batch of texts."""

    # 1. Tokenize the input texts
    tokenized_inputs = tokenizer(texts,
                                 padding=True,
                                 truncation=True,
                                 max_length=max_length,
                                 return_tensors='pt')

    # Move inputs to the same device as the model
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    input_ids = tokenized_inputs['input_ids'].to(device)
    attention_mask = tokenized_inputs['attention_mask'].to(device)

    # 2. Set the model to evaluation mode
    model.to(device) # Ensure model is on the correct device
    model.eval()

    # 3. Perform inference
    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits

    # 4. Apply softmax to get probabilities
    probabilities = F.softmax(logits, dim=1)

    # 5. Extract the probability for the positive class (assuming index 1 is positive)
    # The order of labels is usually determined during training, here we assume 0: Negative, 1: Positive
    positive_probabilities = probabilities[:, 1].cpu().numpy()

    return positive_probabilities

print("Sentiment prediction function 'predict_sentiment_batch' defined.")

Sentiment prediction function 'predict_sentiment_batch' defined.


In [ ]:
import pandas as pd
import numpy as np

# Initialize lists to store the results
num_comments_list = []
avg_sentiment_list = []

print("Processing sentiment for news article comments...")

# Iterate through each row of the DataFrame
for index, row in df.iterrows():
    # Extract comments from the 'comments' column
    raw_comments = row['comments']

    # Filter out empty or non-string comments and extract the 'content' field
    comment_texts = [c.get('content', '') for c in raw_comments if c and c.get('content')]

    num_comments = len(comment_texts)
    num_comments_list.append(num_comments)

    if num_comments > 0:
        # Predict sentiment for the batch of comments
        # Using a batch size to prevent memory issues for articles with many comments
        batch_size = 32 # Adjust based on available memory
        all_positive_probs = []
        for i in range(0, num_comments, batch_size):
            batch_texts = comment_texts[i:i+batch_size]
            positive_probs = predict_sentiment_batch(batch_texts, tokenizer, loaded_sentiment_model)
            all_positive_probs.extend(positive_probs)

        # Calculate the average positive sentiment probability
        avg_sentiment_list.append(np.mean(all_positive_probs))
    else:
        # If no comments, sentiment is NaN or 0, depending on desired representation
        avg_sentiment_list.append(np.nan) # Using NaN for no comments

# Add the new columns to the DataFrame
df['num_comments'] = num_comments_list
df['avg_comment_sentiment'] = avg_sentiment_list

print("Comment sentiment analysis complete. Displaying updated DataFrame head:")
display(df.head())

Processing sentiment for news article comments...
Comment sentiment analysis complete. Displaying updated DataFrame head:


,content,comments,news_category,num_comments,avg_comment_sentiment
0,大宗商品将创下自2016年以来的最佳年度表现，原油到铜的年度收益将有所增长。 彭博商品指数创...,[],财经,0,NaN
1,国泰君安党委书记贺青：奋进新时代 共建持续健康繁荣的资本市场 贺青 国泰君安证券 “党的十九...,[],财经,0,NaN
2,原标题：穿越大江大河 时光不恋过往，大江永远奔腾。你好，2020年！你好，下一个十年！ 当2...,[],教育,0,NaN
3,新浪美股1月1日讯 美国总统特朗普曾经强调，被新减税政策选中的低收入地区房价“飙升”，证明他...,"[{'area': '福建泉州', 'content': '看看消费水平就知道了，啥也别说了...",科技,1,0.992747
4,原标题：新年前一天，林郑在警总告诉警队：相信正义站在我们一边！ 林郑月娥说，香港这支卓越的警...,"[{'area': '湖北黄石', 'content': '香港的明天会更加美好！', 'n...",时政,60,0.960618


经过一段时间的加载与判断后，就此完成了新闻数据中评论的情感划分，并且对于有复数评论的新闻，则计算该新闻的情感值的平均，进而观测该新闻下整体评论的情绪倾向。


In [ ]:
display(df.head(10))

,content,comments,news_category,num_comments,avg_comment_sentiment
0,大宗商品将创下自2016年以来的最佳年度表现，原油到铜的年度收益将有所增长。 彭博商品指数创...,[],财经,0,NaN
1,国泰君安党委书记贺青：奋进新时代 共建持续健康繁荣的资本市场 贺青 国泰君安证券 “党的十九...,[],财经,0,NaN
2,原标题：穿越大江大河 时光不恋过往，大江永远奔腾。你好，2020年！你好，下一个十年！ 当2...,[],教育,0,NaN
3,新浪美股1月1日讯 美国总统特朗普曾经强调，被新减税政策选中的低收入地区房价“飙升”，证明他...,"[{'area': '福建泉州', 'content': '看看消费水平就知道了，啥也别说了...",科技,1,0.992747
4,原标题：新年前一天，林郑在警总告诉警队：相信正义站在我们一边！ 林郑月娥说，香港这支卓越的警...,"[{'area': '湖北黄石', 'content': '香港的明天会更加美好！', 'n...",时政,60,0.960618
5,原标题：驻伊拉克使馆被“攻破”，特朗普大骂伊朗！ 特朗普：伊朗正在策划对美国驻伊拉克大使馆的...,[],教育,0,NaN
6,原标题：创新高！2019年中国内地票房642.66亿元 17.27亿人次观影 中新经纬客户端...,"[{'area': '广东广州', 'content': '我和我的祖国，除了包场和刷票，估...",娱乐,7,0.877209
7,原标题：创新高！2019年中国内地票房642.66亿元 17.27亿人次观影 中新经纬客户端...,[],娱乐,0,NaN
8,原标题：2020年起，这些新规正式施行，“医”食住行都涵盖 中新经纬客户端1月1日电 （熊思...,"[{'area': '湖南', 'content': '2020演唱会🙏', 'nickna...",时政,1,0.992979
9,原标题：2019年人民币汇率中间价调贬1130基点 今年或小幅升值 中新经纬客户端1月1日电...,"[{'area': '香港', 'content': '赶紧破8，不然就来不及了。', 'n...",财经,2,0.987838


##### 查看评论偏负向的新闻

In [ ]:
# Filter DataFrame for news with average comment sentiment less than 0.5
low_sentiment_news = df[df['avg_comment_sentiment'] < 0.5]

print("News articles with average comment sentiment less than 0.5:")
display(low_sentiment_news.head(10)) # Display the first 10 rows of the filtered DataFrame

print(f"Total number of news articles with average comment sentiment less than 0.5: {len(low_sentiment_news)}")

News articles with average comment sentiment less than 0.5:


,content,comments,news_category,num_comments,avg_comment_sentiment
403,央行：引导金融机构加大对小微、民营企业的支持力度 央行召开第四季度例会。会议认为，深化金融供...,"[{'area': '重庆', 'content': '研讨交流中，与会专家围绕“安全形势与...",财经,2,0.352327
449,报告称2019年12月全国100城新建住宅均价环比上涨0.42% 作者：王舒嫄 中证网讯（记...,"[{'area': '黑龙江哈尔滨', 'content': '中国的自信缘于中国找到了一条...",科技,1,0.028316
747,“这是一场电影里才有的逃亡”！日产前董事长戈恩从日本“逃走”，留下一堆谜团 2019年的倒数...,"[{'area': '山西吕梁', 'content': '[吐][吐][吐]', 'nic...",时政,1,0.021906
920,来源：金十数据 金十数据 免责声明：自媒体综合提供的内容均源自自媒体，版权归原作者所有，转载...,"[{'area': '北京', 'content': '这可真是太秀了[允悲]', 'nic...",财经,1,0.021935
968,热点栏目 自选股 数据中心 行情中心 资金流向 模拟交易 客户端 来源：方正中期期货有限公...,"[{'area': '河北保定', 'content': '上午鸡蛋多方还放量拉升了，结果下...",财经,1,0.027713
1438,来源：经济参考报 记者 钟源 北京报道 受中小企业融资需求巨大和打新资金“青睐”的影响，我国...,"[{'area': '陕西西安', 'content': '【陕西安康：一涉黑案主犯被判十八...",财经,9,0.237933
1444,原标题：央行开年发出“全面降准”礼包 2020年货币政策操作仍有较大空间 来源：经济参考报 ...,"[{'area': '河南开封', 'content': '美国可以随意杀人，但他国只要不听...",财经,1,0.492121
1529,热点栏目 自选股 数据中心 行情中心 资金流向 模拟交易 客户端 新浪财经讯 1月2日贵州...,"[{'area': '广东广州', 'content': '現存的景色固然是好，但那些被人為...",财经,3,0.433680
1537,文/熊光的朋友 看似“升级”，实为“换血”。阿里旗下唯一的智能硬件产品线——天猫精灵，正遭受...,"[{'area': '广东深圳', 'content': '愤怒的伊拉克人估计什么事情都干得...",家居,1,0.022023
1988,视频加载中，请稍候... 自动播放 play 粤开策略聊-2020年A股年度策略第三弹 向前...,"[{'area': '浙江衢州', 'content': '判决赔一元是啥意思？换个账号继续...",财经,2,0.307291


Total number of news articles with average comment sentiment less than 0.5: 949


In [ ]:
news_index = 1537

In [ ]:
print(df.loc[news_index, 'content'])

文/熊光的朋友 看似“升级”，实为“换血”。阿里旗下唯一的智能硬件产品线——天猫精灵，正遭受着新一轮内外部的双重施压。 2020年第一天，阿里突然放出消息，将人工智能实验室天猫精灵业务“升级”为独立事业部，由阿里云IoT负责人库伟负责该事业部。 而天猫精灵业务原负责人陈丽娟（花名：浅雪）将带领人工智能实验室其余业务加入阿里云智能事业群，负责产品解决方案与大网站事业部，推动并建立云智能2B产品体系。 虽然整个消息内容显得“暧昧不清”，但我们仍然能获得两个有效信息： 天猫精灵智能音箱业务线在推出将近3年后，首次大换血，人工智能实验室“创始人”陈丽娟将另有任用。 天猫精灵事业部的新负责人库伟，也将继续负责阿里云IoT事业部，这个部门主要做的事，是为硬件制造业提供智能化改造技术方案与云端服务。两个事业部是否融合虽然没有定论，但这条智能音箱业务线兜兜转转，最终又回到了阿里云事业群，甚至可能成为IoT事业部的一部分。 虽然此次以“升级”为名进行了人士调整，但回看近一年来，天猫精灵与其背后的人工智能实验室在内部架构调整中“地位”的起起落落，不难看出，这个阿里设立不到3年的年轻业务正在经历着一段阵痛期。 人工智能实验室前负责人浅雪 在张勇于2018年9月接任阿里巴巴董事局主席后，同年11月，阿里的内部架构就进行了一次洗牌式调整。 其中，于2016年低调孵化于阿里云内部，并在2017年7月正式向外界推出智能音箱品牌——“天猫精灵”的人工智能实验室，不再隶属于阿里云，而是被纳入集团创新业务事业群。实验室负责人陈丽娟，改为直接向CEO张勇汇报。 1年后，也就是2019年6月，张勇再次发“全员信”进行业务重组，首当其中被调整的便是“阿里创新业务群”—— 原阿里大文娱新媒体业务总裁朱顺炎出任总裁，陈丽娟将向他进行汇报；阿里文学、阿里音乐以及UC及旗下移动创新业务都被归到创新业务群。 让人疑惑的是，作为一款硬件设备，“天猫精灵”夹杂在偏内容娱乐方向的业务板块里面，显得有些特立独行。 当然，这次更引人注目的，是汇报关系上的直接降级。这一变化甚至曾被外界猜测为“阿里将放弃智能音箱业务”。 又过了半年，如今，除了做出“换帅”这个重大决定，阿里让天猫精灵重新拥抱云与IOT（物联网），倒像是回到了4年前智能音箱诞生时，虽然老套但却最原始的目标上——做智能家居网络的控制中枢。 或者说，做IOT时代的超级入

In [ ]:
news_index = 1438
for i in range(len(df.loc[news_index, 'comments'])):
  print(df.loc[news_index, 'comments'][i]['content'])

【陕西安康：一涉黑案主犯被判十八年】
嗯嗯
【陕西安康：一涉黑案主犯被判十八年】
【陕西安康：一涉黑案主犯被判十八年】
都2020年了 那些人还以为自己活在七八十年代呢
【陕西安康：一涉黑案主犯被判十八年】
【陕西安康：一涉黑案主犯被判十八年】
【陕西安康：一涉黑案主犯被判十八年】
30余人持械打砸KTV 陕西安康：一涉黑案主犯被判十八年


#### 分析数据存储
下面将分析后的数据存储为news_with_sentiment.json文件中，以便之后在不对训练模型修改的情况下可以直接加载模型分析后的数据进行进一步的分析。

In [ ]:
import os

output_filename = 'news_with_sentiment.json'

# Save the DataFrame to JSON
df.to_json(output_filename, orient='records', force_ascii=False, indent=4)

# Check if the file exists before trying to get its size
if os.path.exists(output_filename):
    print(f"DataFrame successfully saved to '{output_filename}'")
    print(f"File size: {os.path.getsize(output_filename) / (1024*1024):.2f} MB")
else:
    print(f"Error: The file '{output_filename}' was not found after the save operation. Please check the current working directory and permissions.")

DataFrame successfully saved to 'news_with_sentiment.json'
File size: 898.64 MB


#### 加载分析数据

下面将存储的数据加载，并做一些简单的数据分析。

In [6]:
import pandas as pd

# Define the filename
output_filename = 'news_with_sentiment.json'

# Load the JSON file into a DataFrame
df_loaded = pd.read_json(output_filename)

##### 筛选出待评论的新闻


In [18]:
df_filtered_comments = df_loaded[df_loaded['num_comments'] > 0]

print(f"Original DataFrame shape: {df_loaded.shape}")
print(f"Filtered DataFrame shape (articles with comments): {df_filtered_comments.shape}")

print("Head of the filtered DataFrame:")
display(df_filtered_comments.head())

Original DataFrame shape: (129969, 5)
Filtered DataFrame shape (articles with comments): (59364, 5)
Head of the filtered DataFrame:


,content,comments,news_category,num_comments,avg_comment_sentiment
3,新浪美股1月1日讯 美国总统特朗普曾经强调，被新减税政策选中的低收入地区房价“飙升”，证明他...,"[{'area': '福建泉州', 'content': '看看消费水平就知道了，啥也别说了...",科技,1,0.992747
4,原标题：新年前一天，林郑在警总告诉警队：相信正义站在我们一边！ 林郑月娥说，香港这支卓越的警...,"[{'area': '湖北黄石', 'content': '香港的明天会更加美好！', 'n...",时政,60,0.960618
6,原标题：创新高！2019年中国内地票房642.66亿元 17.27亿人次观影 中新经纬客户端...,"[{'area': '广东广州', 'content': '我和我的祖国，除了包场和刷票，估...",娱乐,7,0.877209
8,原标题：2020年起，这些新规正式施行，“医”食住行都涵盖 中新经纬客户端1月1日电 （熊思...,"[{'area': '湖南', 'content': '2020演唱会🙏', 'nickna...",时政,1,0.992979
9,原标题：2019年人民币汇率中间价调贬1130基点 今年或小幅升值 中新经纬客户端1月1日电...,"[{'area': '香港', 'content': '赶紧破8，不然就来不及了。', 'n...",财经,2,0.987838


##### 新增评论倾向性列

将平均情感值大于0.5的新闻划分为正向评论新闻，意味着评论平均偏向正向，反之则负向。

In [19]:
df_filtered_comments['comment_positivity'] = df_filtered_comments['avg_comment_sentiment'].apply(lambda x: 1 if x > 0.5 else 0)

print("DataFrame with 'comment_positivity' column added:")
display(df_filtered_comments.head())

DataFrame with 'comment_positivity' column added:


/tmp/ipython-input-2420334799.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered_comments['comment_positivity'] = df_filtered_comments['avg_comment_sentiment'].apply(lambda x: 1 if x > 0.5 else 0)


,content,comments,news_category,num_comments,avg_comment_sentiment,comment_positivity
3,新浪美股1月1日讯 美国总统特朗普曾经强调，被新减税政策选中的低收入地区房价“飙升”，证明他...,"[{'area': '福建泉州', 'content': '看看消费水平就知道了，啥也别说了...",科技,1,0.992747,1
4,原标题：新年前一天，林郑在警总告诉警队：相信正义站在我们一边！ 林郑月娥说，香港这支卓越的警...,"[{'area': '湖北黄石', 'content': '香港的明天会更加美好！', 'n...",时政,60,0.960618,1
6,原标题：创新高！2019年中国内地票房642.66亿元 17.27亿人次观影 中新经纬客户端...,"[{'area': '广东广州', 'content': '我和我的祖国，除了包场和刷票，估...",娱乐,7,0.877209,1
8,原标题：2020年起，这些新规正式施行，“医”食住行都涵盖 中新经纬客户端1月1日电 （熊思...,"[{'area': '湖南', 'content': '2020演唱会🙏', 'nickna...",时政,1,0.992979,1
9,原标题：2019年人民币汇率中间价调贬1130基点 今年或小幅升值 中新经纬客户端1月1日电...,"[{'area': '香港', 'content': '赶紧破8，不然就来不及了。', 'n...",财经,2,0.987838,1


##### 筛选出大于十条评论且评论倾向负面的新闻

In [21]:
filtered_data = df_filtered_comments[(df_filtered_comments['num_comments'] > 10) & (df_filtered_comments['comment_positivity'] == 0)]

print("News articles with more than 10 comments and 'Negative' positivity reaction:")
display(filtered_data)

News articles with more than 10 comments and 'Negative' positivity reaction:


,content,comments,news_category,num_comments,avg_comment_sentiment,comment_positivity
8995,原标题：“中国第一高楼”在内的多处绿地项目被曝欠债停工，负债9成的绿地金控业务负重前行，为锦...,"[{'area': '北京', 'content': ' 【 失踪女童安全】', 'ni...",财经,22,0.389624,0
20991,原标题：韩艺人生日，中国粉丝给韩军送礼？ 1月12日是正在服兵役的韩国艺人、韩国男团EXO成...,"[{'area': '奥地利', 'content': '虽然咱本来就知道“轻伤”是入刑的法...",娱乐,25,0.492330,0
21739,原标题：山东张志超案再审改判无罪 昨日宣判后，张志超（右二）和律师们在法庭外合影。 受访者供...,"[{'area': '广东广州', 'content': '话说，这个女孩子一个月前就失踪了...",时政,13,0.447851,0
22488,视频加载中，请稍候... 自动播放 play 暖冬叠加早春 开门红行情仍将延续 向前 向后 ...,"[{'area': '上海', 'content': '【 知道真相后令人心疼[话筒]】'...",财经,12,0.419254,0
24616,新酷产品第一时间免费试玩，还有众多优质达人分享独到生活经验，快来新浪众测，体验各领域最前沿...,"[{'area': '河南平顶山', 'content': '严厉打击乱港分子', 'nic...",科技,11,0.462363,0
53976,浙商银行“绿色通道”2小时快速汇出防疫跨境采购资金 中证网讯（记者 戴安琪）1月30日下午，...,"[{'area': '安徽安庆', 'content': '天不佑好人，痛心！[泪][泪][...",时政,12,0.449695,0
55375,原标题：2人销售伪劣口罩在太原落网 涉案金额达600余万 2月1日，太原警方协助北京警方，经...,"[{'area': '北京海淀', 'content': '奇怪那个玩手机的小伙，蹲局子能玩...",财经,15,0.474725,0
61208,分析师Kimble指出，毋庸置疑，美国银行的健康状况对该国经济至关重要。这一点也反映在股市上...,"[{'area': '上海', 'content': '愚昧的农民 就这认识程度 还得穷三代...",财经,18,0.493668,0
68293,视频加载中，请稍候... 自动播放 play 向前 向后 原标题：富士康推迟复工，特斯拉10...,"[{'area': '浙江杭州', 'content': '太可惜了！', 'nicknam...",时政,200,0.477968,0
68493,原标题：经开区将为企业提供公租房、蓝领公寓 保障返京员工住宿 新京报快讯（记者 陈琳）节后企...,"[{'area': '上海', 'content': '这些人发国难财没人性了。', 'ni...",时政,36,0.413259,0


In [28]:
df_loaded.loc[24616, 'content']

' 新酷产品第一时间免费试玩，还有众多优质达人分享独到生活经验，快来新浪众测，体验各领域最前沿、最有趣、最好玩的产品吧~！下载客户端还能获得专享福利哦！ 2009年10月22日诞生至今，历经10年，这一天终于还是来了。 因易用、简单、使用高效，Windows 7成为Windows XP之后的又一“常青树”，它一改Vista的颓败，成为Windows 历史上最好用的版本之一。 但世间没有不散的宴席，技术总在不断更迭，随着更安全、更高效、功能更强大的Windows 10面世，Windows 7完成了历史使命。 2020年1月14日，也就是今天，微软正式停止对Windows 7的支持。 其实，早在2015年1月13日，微软就终止了对Windows 7SP1的主流技术支持，此后仅发放安全补丁和紧急修复补丁。5年的缓冲期转瞬即逝，如今最后一道“保险”也没了。 社会总在不停发展，再美好的事物总会老去，随着时间推移，被更先进、更安全、更好用的代替，历史规律亘古不变。 和当初XP一样，很多人对于Windows 7的情怀终将会随着历史长河消失殆尽。但毫无疑问，伴随我们走过十年的美好回忆值得铭记。 按照微软官方说法，从今天之后，你的电脑还可以继续运行Windows 7，但后果是没有持续软件和安全更新，电脑遭受病毒和恶意软件攻击的风险会更大，主要核心有三点：1、得不到任何问题的技术支持；2、没有软件更新；3、不再提供安全更新或修复。 与此同时，微软官方还解答了一些老用户关心的问题： Windows 7电脑在2020年1月14日会停止工作吗？ 不会。对Windows 7的支持虽然会停止，依然能够能够正常运行。 如何获得Windows 10？ 您可以购买完整版的Windows 10，安装到Windows 7电脑。而使用新电脑是体验Windows 10的最好方式。如今的电脑运行速度更快，功能更强大，而且已经预装了Windows 10。 是否能免费将Windows 7升级为Windows 10？ Windows 10首次发布时，推出过免费升级活动，但该活动已于2016年7月29日结束。 现在你可以购买Windows 10，在原来的老电脑上下载安装。不过，如果你的电脑已使用了三年以上，是合适的时候考虑升级到新设备了。 为什么应该考虑换新电脑？ 最初运行Windows 7的电脑所采用的技术已有10年

In [26]:
for i in range(len(df_loaded.loc[24616, 'comments'])):
  print(df_loaded.loc[24616, 'comments'][i]['content'])

严厉打击乱港分子
自作自受，香港人太宠着了！
转送伊朗，人家正需要！[可怜]
香港暴徒乱港开始向恐怖主义袭击转移了，提高警惕。抓住了，判重刑威慑。
今后港警要争取发现一批暴徒，击毙一批暴徒！
对这样的暴徒根本不需要手软，他们已经要你命，你还管他生死干吗？毙了才不会有危险，才叫人放心。
将这些鬼东西列为恐怖分子，从重严打
作死的节奏
应该判决几个死刑杀鸡儆猴.才能一劳永逸.
这是要大规模杀伤人群吗？要进行大规模无差别谋杀？
这些家伙已和恐怖分子一样丧尽天良，已经不能算是简单的暴力分子，香港的法官即使想替他们开脱恐怕也无能为力了吧？如果这种人都能假释，香港就没有法律可言了。
